# SVD Matrix Factorization (Temporal Split - 99% Coverage)

## Modified Strategy

This notebook uses **temporal split data** BUT filters the test set to achieve **99% coverage**:
- **Training**: 80% oldest ratings (same as original temporal split)
- **Testing**: From 20% newest ratings, **filter to only user-movie pairs where BOTH exist in training**

**Purpose**: Compare algorithm performance on temporal data WITHOUT cold-start complications.

**Key Difference from Original Temporal Split**:
- Original: 8.97% coverage (89% cold-start users)
- This version: ~99% coverage (filters out cold-start pairs)

**Note**: This is NOT realistic for production, but allows fair algorithm comparison on temporal data.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, lil_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print("Using scipy.sparse.linalg.svds for SVD implementation")

## 2. Load Temporal Split Data

In [ ]:
# Load temporal split data
train_path = '../../datasets/output/split_and_train_datasets/temporal_split/train_ratings.csv'
test_path = '../../datasets/output/split_and_train_datasets/temporal_split/test_ratings.csv'

print("=" * 60)
print("LOADING TEMPORAL SPLIT DATA (FILTERED FOR 99% COVERAGE)")
print("=" * 60)
print("\nStrategy: Temporal split + coverage filter")
print("  - Train: 80% oldest ratings")
print("  - Test: 20% newest ratings (FILTERED to known user-movie pairs)")
print("  - Purpose: Fair algorithm comparison on temporal data")
print()

print("Loading training data...")
train = pd.read_csv(train_path)
print(f"Train shape: {train.shape}")

print("\nLoading test data...")
test = pd.read_csv(test_path)
print(f"Test shape: {test.shape}")

print("\nData loaded successfully!")
print("=" * 60)

In [ ]:
# Dataset statistics
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

print("\nTraining Set:")
print(f"  Unique users: {train['userId'].nunique():,}")
print(f"  Unique movies: {train['movieId'].nunique():,}")
print(f"  Total ratings: {len(train):,}")
print(f"  Mean rating: {train['rating'].mean():.2f}")

print("\nTest Set (BEFORE filtering):")
print(f"  Unique users: {test['userId'].nunique():,}")
print(f"  Unique movies: {test['movieId'].nunique():,}")
print(f"  Total ratings: {len(test):,}")
print(f"  Mean rating: {test['rating'].mean():.2f}")

## 3. Create User-Item Matrix

In [ ]:
# Create ID mappings from TRAINING data only
print("Creating ID mappings from TRAINING data...")

unique_users = train['userId'].unique()
unique_movies = train['movieId'].unique()

user_id_map = {id: idx for idx, id in enumerate(unique_users)}
movie_id_map = {id: idx for idx, id in enumerate(unique_movies)}

idx_to_user = {idx: id for id, idx in user_id_map.items()}
idx_to_movie = {idx: id for id, idx in movie_id_map.items()}

print(f"Training users: {len(user_id_map):,}")
print(f"Training movies: {len(movie_id_map):,}")

# Map train and test data to indices
train['user_idx'] = train['userId'].map(user_id_map)
train['movie_idx'] = train['movieId'].map(movie_id_map)

test['user_idx'] = test['userId'].map(user_id_map)
test['movie_idx'] = test['movieId'].map(movie_id_map)

In [ ]:
# CRITICAL: Filter test set to only user-movie pairs that exist in training
print("\n" + "=" * 60)
print("FILTERING TEST SET FOR 99% COVERAGE")
print("=" * 60)

test_before_filter = len(test)
testable = test.dropna(subset=['user_idx', 'movie_idx']).copy()
test_after_filter = len(testable)

print(f"\nTest ratings before filter: {test_before_filter:,}")
print(f"Test ratings after filter: {test_after_filter:,}")
print(f"Removed (cold-start): {test_before_filter - test_after_filter:,}")
print(f"Coverage achieved: {test_after_filter/test_before_filter*100:.2f}%")

# Store for later use
true_coverage = test_after_filter / test_before_filter * 100
total_testable = test_after_filter

print(f"\n✓ Test set filtered to {total_testable:,} testable ratings")

In [ ]:
# Create sparse user-item matrix
print("\nCreating sparse user-item matrix...")

user_item_matrix = csr_matrix(
    (train['rating'].values,
     (train['user_idx'].values, train['movie_idx'].values)),
    shape=(len(user_id_map), len(movie_id_map))
)

print(f"Matrix shape: {user_item_matrix.shape}")
print(f"Memory: {user_item_matrix.data.nbytes / (1024**2):.2f} MB")
print(f"Sparsity: {100 * (1 - user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1])):.2f}%")

# Calculate global mean for filling predictions
global_mean = train['rating'].mean()
print(f"\nGlobal mean rating: {global_mean:.3f}")

## 4. Mean-Centering

In [ ]:
print("=" * 60)
print("MEAN-CENTERING THE RATING MATRIX")
print("=" * 60)

# Compute user means (only over rated items)
print("Computing user means (only over rated items)...")

user_means = np.zeros(user_item_matrix.shape[0])

for user_idx in tqdm(range(user_item_matrix.shape[0]), desc="Computing user means"):
    user_ratings = user_item_matrix.getrow(user_idx).data
    if len(user_ratings) > 0:
        user_means[user_idx] = user_ratings.mean()
    else:
        user_means[user_idx] = global_mean

print(f"\nUser means computed:")
print(f"  Min: {user_means.min():.3f}")
print(f"  Max: {user_means.max():.3f}")
print(f"  Average: {user_means.mean():.3f}")

print("\nMean-centering matrix...")
user_item_matrix_lil = user_item_matrix.tolil()

for user_idx in tqdm(range(user_item_matrix.shape[0]), desc="Centering users"):
    row = user_item_matrix_lil.rows[user_idx]
    data = user_item_matrix_lil.data[user_idx]
    
    if len(row) > 0:
        user_mean = user_means[user_idx]
        for i in range(len(data)):
            data[i] -= user_mean

user_item_matrix_centered = user_item_matrix_lil.tocsr()

print(f"\n✓ Matrix centered successfully")
print(f"  Centered mean: {user_item_matrix_centered.data.mean():.6f} (should be ~0)")

## 5. SVD Decomposition

In [ ]:
# SVD Hyperparameters
K = 50  # Number of latent factors

print("=" * 60)
print("SVD DECOMPOSITION")
print("=" * 60)
print(f"Latent factors (k): {K}")
print(f"\nDecomposing into:")
print(f"  U: {user_item_matrix.shape[0]:,} users × {K} factors")
print(f"  Sigma: {K} singular values")
print(f"  Vt: {K} factors × {user_item_matrix.shape[1]:,} movies")
print("\nThis may take 2-5 minutes...")

start_time = time.time()

# Use which='LM' to get LARGEST singular values
U, sigma, Vt = svds(user_item_matrix_centered, k=K, which='LM')

training_time = time.time() - start_time

print(f"\n✓ SVD completed in {training_time/60:.2f} minutes")
print(f"\nDecomposed matrices:")
print(f"  U shape: {U.shape}")
print(f"  Sigma shape: {sigma.shape}")
print(f"  Vt shape: {Vt.shape}")

# Prepare sigma as diagonal matrix for predictions
sigma_diag = np.diag(sigma)
print(f"\n✓ Ready for predictions!")

## 6. Sample Test Set and Generate Predictions

In [ ]:
# Configuration
SAMPLE_SIZE = 100000  # 100K for fair comparison

# Sample from testable ratings
if SAMPLE_SIZE and SAMPLE_SIZE < len(testable):
    print(f"Sampling {SAMPLE_SIZE:,} from {len(testable):,} testable ratings")
    test_sample = testable.sample(SAMPLE_SIZE, random_state=42)
else:
    print(f"Using all {len(testable):,} testable ratings")
    test_sample = testable.copy()

print(f"\nGenerating predictions for {len(test_sample):,} ratings...")

start_time = time.time()

test_predictions = []
test_actuals = []

for idx, row in tqdm(test_sample.iterrows(), total=len(test_sample), desc="Predicting"):
    user_idx = int(row['user_idx'])
    movie_idx = int(row['movie_idx'])
    actual = row['rating']
    
    # Get prediction: U[user] × Sigma × Vt[:,movie]
    user_vector = U[user_idx, :]
    movie_vector = Vt[:, movie_idx]
    
    # Compute centered prediction
    pred_centered = np.dot(user_vector, np.dot(sigma_diag, movie_vector))
    
    # Add back user's mean rating (de-centering)
    pred = pred_centered + user_means[user_idx]
    
    # Clip to valid range
    pred = np.clip(pred, 0.5, 5.0)
    
    test_predictions.append(pred)
    test_actuals.append(actual)

prediction_time = time.time() - start_time

print(f"\n✓ Predictions completed in {prediction_time:.2f} seconds")
print(f"  Average: {prediction_time/len(test_sample)*1000:.3f} ms per rating")

# Convert to numpy arrays
test_actuals = np.array(test_actuals)
test_predictions = np.array(test_predictions)

## 7. Evaluation

In [ ]:
# Calculate metrics
rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
mae = mean_absolute_error(test_actuals, test_predictions)
correlation = np.corrcoef(test_actuals, test_predictions)[0, 1]

print("="*60)
print("SVD RESULTS (TEMPORAL SPLIT - 99% COVERAGE)")
print("="*60)
print(f"\nAlgorithm: SVD Matrix Factorization")
print(f"Split: Temporal (filtered for known user-movie pairs)")
print(f"Implementation: scipy.sparse.linalg.svds")
print(f"Latent factors: {K}")

print(f"\nPerformance:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  Correlation: {correlation:.4f}")

print(f"\nTiming:")
print(f"  Training: {training_time/60:.2f} minutes")
print(f"  Prediction: {prediction_time:.2f} seconds")
print(f"  Per rating: {prediction_time/len(test_sample)*1000:.2f} ms")

print(f"\nCoverage:")
print(f"  Coverage: {true_coverage:.2f}%")
print(f"  Test samples: {len(test_sample):,}")
print(f"  Total testable: {total_testable:,}")
print("="*60)

## 8. Save Results

In [ ]:
# Save results
results = {
    'algorithm': 'SVD (Temporal-99)',
    'split_strategy': 'temporal (filtered)',
    'method': 'Matrix Factorization',
    'implementation': 'scipy.sparse.linalg.svds',
    'n_factors': K,
    'rmse': rmse,
    'mae': mae,
    'coverage': true_coverage,
    'training_time_minutes': training_time/60,
    'prediction_time_ms': prediction_time/len(test_sample)*1000,
    'test_samples': len(test_sample),
    'total_testable': total_testable,
    'correlation': correlation
}

results_df = pd.DataFrame([results])
output_path = '../../datasets/output/model_implementations/svd_temporal_99_coverage.csv'
results_df.to_csv(output_path, index=False)

print(f"✓ Results saved to: {output_path}")
print("\nResults Summary:")
print(results_df.T)

## Summary

This notebook evaluates SVD on **temporal split data with 99% coverage**.

**Key Points**:
- Uses temporal train/test split (time-ordered)
- Filters test set to only known user-movie pairs
- Achieves ~99% coverage (vs 9% in original temporal split)
- Allows fair algorithm comparison on temporal data

**Comparison Purpose**:
- Compare with 80-20 random split (different split, same coverage)
- Shows if temporal ordering affects algorithm performance
- Isolates algorithm quality from cold-start effects